# 00: Token Mechanics, KV-Cache Physics & Roofline Hardware Ceilings

> **Curriculum Phase**: `Phase 00 - Foundations & Token Mechanics`  
> **Prerequisites**: Python 3.12+, basic understanding of transformer inference  
> **Key Concepts**: Byte-Pair Encoding (BPE), Subword Tokenization Ratios, KV-Cache Memory Bandwidth, Attention Architectures (MHA, GQA, MLA), and the Hardware Roofline Model.  

Welcome to the interactive companion notebook for **Phase 00: Foundations and Token Mechanics**. In this hands-on lab, you will move beyond high-level prompt abstractions and explore the physical mechanics of Large Language Models:

1. **Tokenization Mechanics**: How Byte-Pair Encoding (BPE) turns UTF-8 bytes into subword tokens, and why multi-lingual texts or formatted numbers suffer from the tokenization tax.
2. **KV-Cache VRAM & Bandwidth Sizing**: Deriving exact GPU memory allocations across Multi-Head Attention (MHA), Grouped-Query Attention (GQA), and Multi-Head Latent Attention (MLA) across 4K, 32K, and 128K context windows.
3. **The Inference Roofline Model**: Plotting memory-bandwidth vs compute ceilings to understand why prefill is compute-bound while autoregressive token generation (decode) is memory-bandwidth bound.

In [ ]:
# %% [Setup] Colab & Environment Bootstrap
import os, sys
try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    print('Running in Google Colab.')
    repo_dir = '/content/repo'
    if not os.path.exists(repo_dir):
        print('Cloning repository...')
        !git clone --depth 1 https://github.com/dip4k/senior-engineer-to-ai-engineer-roadmap.git /content/repo
    %cd /content/repo
    !pip install -q --no-warn-conflicts -r agent-forge/requirements.txt -r notebooks/requirements-notebooks.txt
    if os.path.abspath('agent-forge') not in sys.path:
        sys.path.insert(0, os.path.abspath('agent-forge'))
else:
    # Local environment setup: locate repo root and agent-forge
    repo_root = os.path.abspath('..') if os.path.exists('../agent-forge') else os.path.abspath('.')
    af_path = os.path.join(repo_root, 'agent-forge')
    if os.path.exists(af_path) and af_path not in sys.path:
        sys.path.insert(0, af_path)
    print(f'Running locally with agent-forge in path: {af_path}')

### Environment Setup & Dependencies

We import standard analytical and visualization libraries. If running in Google Colab, the bootstrap cell above has cloned the repository and installed all requirements.

In [ ]:
import math
import re
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

# Configure clean visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print('Dependencies loaded successfully.')

## Part 1: Tokenization Mechanics: Byte-Pair Encoding (BPE) from Scratch

A transformer model never processes raw characters or strings. It receives an array of integers representing discrete vocabulary entries. Modern tokenizers use **Byte-Pair Encoding (BPE)** (Sennrich et al., 2015), which builds a subword vocabulary by iteratively counting adjacent symbol pairs and merging the most frequent pair into a new symbol.

### How BPE Training Works:
1. **Initialization**: Start with a base vocabulary of single symbols (e.g. all 256 byte values or base characters).
2. **Frequency Counting**: Count every adjacent pair of symbols across the training corpus.
3. **Merge Step**: Find the most frequent pair, replace all occurrences in the corpus with a newly created merged token, and append this rule to the merge table.
4. **Iterate**: Repeat until the target vocabulary size or merge count is reached.

At inference time, the tokenizer applies these learned merges in exact historical order to segment any arbitrary string into subwords.

In [ ]:
class ToyBPETokenizer:
    """An educational Byte-Pair Encoding (BPE) tokenizer demonstrating subword splitting."""

    def __init__(self, target_merges: int = 15):
        self.target_merges = target_merges
        self.merges: List[Tuple[str, str, str]] = []  # (pair_0, pair_1, merged_token)
        self.vocab: Dict[int, str] = {}
        self.token_to_id: Dict[str, int] = {}

    def get_stats(self, word_freqs: Dict[Tuple[str, ...], int]) -> Dict[Tuple[str, str], int]:
        pairs: Dict[Tuple[str, str], int] = {}
        for word_tuple, freq in word_freqs.items():
            for i in range(len(word_tuple) - 1):
                pair = (word_tuple[i], word_tuple[i + 1])
                pairs[pair] = pairs.get(pair, 0) + freq
        return pairs

    def merge_word(self, word_tuple: Tuple[str, ...], pair: Tuple[str, str], new_token: str) -> Tuple[str, ...]:
        new_word = []
        i = 0
        while i < len(word_tuple):
            if i < len(word_tuple) - 1 and word_tuple[i] == pair[0] and word_tuple[i + 1] == pair[1]:
                new_word.append(new_token)
                i += 2
            else:
                new_word.append(word_tuple[i])
                i += 1
        return tuple(new_word)

    def train(self, corpus: str):
        # Split words into characters with end-of-word marker </w>
        words = corpus.strip().split()
        word_freqs: Dict[Tuple[str, ...], int] = {}
        for w in words:
            symbols = tuple(list(w) + ["</w>"])
            word_freqs[symbols] = word_freqs.get(symbols, 0) + 1

        # Initialize vocabulary with unique characters
        base_vocab = set()
        for symbols in word_freqs.keys():
            base_vocab.update(symbols)

        current_id = 0
        for sym in sorted(base_vocab):
            self.vocab[current_id] = sym
            self.token_to_id[sym] = current_id
            current_id += 1

        print(f"Initial base character vocabulary size: {len(self.vocab)}")

        # Iteratively find and merge most frequent pairs
        for step in range(self.target_merges):
            pairs = self.get_stats(word_freqs)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            count = pairs[best_pair]
            new_token = best_pair[0] + best_pair[1]
            self.merges.append((best_pair[0], best_pair[1], new_token))
            self.vocab[current_id] = new_token
            self.token_to_id[new_token] = current_id
            current_id += 1

            # Update corpus word representations
            new_word_freqs = {}
            for w_tuple, freq in word_freqs.items():
                merged = self.merge_word(w_tuple, best_pair, new_token)
                new_word_freqs[merged] = freq
            word_freqs = new_word_freqs

        print(f"Completed {len(self.merges)} merges. Total vocabulary size: {len(self.vocab)}")

    def encode(self, text: str) -> List[str]:
        tokens = []
        for w in text.strip().split():
            w_tuple = tuple(list(w) + ["</w>"])
            for p0, p1, merged in self.merges:
                w_tuple = self.merge_word(w_tuple, (p0, p1), merged)
            tokens.extend(list(w_tuple))
        return tokens

# Train Toy BPE on representative domain corpus
corpus = (
    "low lower lowest newest widest "
    "unbreakable break breaks breaking unbreak "
    "lower lower lowest lowest newest "
    "widest break breaking unbreakable"
)
tokenizer = ToyBPETokenizer(target_merges=12)
tokenizer.train(corpus)

# Inspect learned merge operations
merge_df = pd.DataFrame(tokenizer.merges, columns=["Pair 0", "Pair 1", "Merged Subword Token"])
merge_df.index = [f"Merge {i+1}" for i in range(len(merge_df))]
display(merge_df)

# Test subword splitting on unseen words
test_words = ["lowest", "unbreakable", "newer", "wider", "breaker"]
print("\n--- Subword Splitting Inference ---")
for tw in test_words:
    pieces = tokenizer.encode(tw)
    print(f"Input: '{tw:<12}' -> Subwords: {pieces}")

### Character vs Byte vs Token Ratio Comparison

Because BPE vocabularies are trained predominantly on English-dominated web crawls, common English words are assigned single token IDs. In contrast, non-English languages, structured code, and formatted numbers are split into multiple fine-grained subwords or raw UTF-8 byte tokens.

This introduces critical production implications:
- **The Non-English Token Tax**: The exact same semantic statement costs 2x to 5x more tokens in Hindi, Japanese, or German than in English.
- **Formatted Numbers**: `1,000,000` often splits into 3–5 tokens (`1`, `,`, `000`, `,`, `000`), whereas `1000000` is 2 tokens.
- **Whitespace Sensitivity**: In modern tokenizers (like OpenAI `cl100k_base` and `o200k_base`), leading spaces are glued to words. `" Hello"` is 1 token, while `"  Hello"` (double space) becomes 2 tokens.

In [ ]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    tokenize_fn = lambda txt: enc.encode(txt)
    tokenizer_name = "tiktoken cl100k_base"
except Exception:
    # Deterministic fallback estimator matching empirical BPE splitting patterns
    def tokenize_fn(txt: str) -> List[int]:
        # Approximate subwords by splitting alphanumeric words and non-ASCII clusters
        subwords = re.findall(r"[\w']+|[\s]+|[^\w\s]", txt, re.UNICODE)
        expanded = []
        for sw in subwords:
            if any(ord(c) > 127 for c in sw):
                # Non-ASCII UTF-8 multibyte expansion (~1.5 to 2 tokens per 3 bytes)
                expanded.extend([1] * max(1, math.ceil(len(sw.encode('utf-8')) / 2)))
            elif len(sw) > 4:
                expanded.extend([1] * math.ceil(len(sw) / 4))
            else:
                expanded.append(1)
        return expanded
    tokenizer_name = "heuristic subword model"

sample_texts = {
    "English": "The refund policy allows returns within thirty days.",
    "Spanish": "La política de reembolso permite devoluciones en un plazo de treinta días.",
    "German": "Die Rückerstattungsrichtlinie ermöglicht Rückgaben innerhalb von dreißig Tagen.",
    "Hindi": "धनवापसी नीति तीस दिनों के भीतर वापसी की अनुमति देती है।",
    "Japanese": "返金ポリシーにより、30日以内の返品が可能です。",
    "Chinese": "退款政策允许在三十天内退货。",
    "Code (camelCase)": "def getUserById(userId: str) -> dict: return {'id': userId}",
    "Code (snake_case)": "def get_user_by_id(user_id: str) -> dict: return {'id': user_id}",
    "Formatted Number": "Total revenue was $1,000,000 in Q4.",
    "Raw Number": "Total revenue was $1000000 in Q4.",
}

data_rows = []
english_tokens = len(tokenize_fn(sample_texts["English"]))

for label, text in sample_texts.items():
    tokens = tokenize_fn(text)
    n_tokens = len(tokens)
    n_chars = len(text)
    n_bytes = len(text.encode('utf-8'))
    expansion = n_tokens / english_tokens if "English" in sample_texts else 1.0
    data_rows.append({
        "Category / Language": label,
        "Characters": n_chars,
        "UTF-8 Bytes": n_bytes,
        "Tokens": n_tokens,
        "Chars / Token": round(n_chars / n_tokens, 2),
        "Bytes / Token": round(n_bytes / n_tokens, 2),
        "Relative Cost (vs English)": f"{expansion:.2f}x"
    })

bench_df = pd.DataFrame(data_rows)
print(f"Tokenizer Active: {tokenizer_name}")
display(bench_df)

In [ ]:
# Visualizing the Multilingual Token Tax
languages = ["English", "Spanish", "German", "Hindi", "Japanese", "Chinese"]
lang_df = bench_df[bench_df["Category / Language"].isin(languages)].copy()
lang_df["Multiplier"] = [float(x.replace('x', '')) for x in lang_df["Relative Cost (vs English)"]]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(lang_df["Category / Language"], lang_df["Tokens"], color=["#2563eb", "#0284c7", "#0d9488", "#dc2626", "#d97706", "#7c3aed"])
ax.set_xlabel("Token Count for Identical Semantic Refund Policy")
ax.set_title("The Tokenization Tax: Semantic Equivalence vs Token Cost Across Languages", fontsize=12, pad=12)
ax.grid(axis='x', linestyle='--', alpha=0.7)

for bar, mult in zip(bars, lang_df["Multiplier"]):
    width = bar.get_width()
    ax.text(width + 0.5, bar.get_y() + bar.get_height() / 2, f"{int(width)} tokens ({mult:.1f}x)",
            va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## Part 2: KV-Cache Memory Bandwidth and VRAM Sizing Calculator

During autoregressive token generation, recomputing Key and Value tensors for all preceding tokens at every step would yield quadratic O(N²) compute latency. The **Key-Value (KV) Cache** stores previous Key and Value projection tensors in GPU High-Bandwidth Memory (HBM).

### The KV-Cache Memory Formula
For a model with `n_layers`, `n_kv_heads`, head dimension `d_head`, precision `bytes_per_elem` (2 for FP16/BF16, 1 for FP8), context length `seq_len`, and batch size `B`:

```text
KV_Cache_Bytes = 2 × n_layers × n_kv_heads × d_head × bytes_per_elem × seq_len × batch_size
```

The factor of 2 accounts for both the **Key** and **Value** tensors.

### Architectural Levers: MHA vs GQA vs MLA
- **Multi-Head Attention (MHA)** (e.g. Llama 2 70B): `n_kv_heads = n_heads = 64`. Maximum VRAM footprint.
- **Grouped-Query Attention (GQA)** (e.g. Llama 3.1 70B): `n_kv_heads = 8`, while query heads = 64. Groups share KV heads, yielding an **8x KV cache reduction**.
- **Multi-Head Latent Attention (MLA)** (e.g. DeepSeek-V2 / V3): Instead of storing full uncompressed keys/values, caches a compressed low-rank latent vector per token (`d_c + d_h_R` = 512 + 64 = 576 dimensions). This delivers a **93.3% KV cache reduction**.

In [ ]:
class ModelConfig(BaseModel):
    name: str
    layers: int = Field(gt=0)
    query_heads: int = Field(gt=0)
    kv_heads: int = Field(gt=0)
    head_dim: int = Field(gt=0)
    param_count_billions: float = Field(gt=0)
    attention_type: str
    mla_latent_dim: int = 0  # Only for MLA

    def kv_bytes_per_token(self, bytes_per_elem: int = 2) -> int:
        if self.attention_type == "MLA":
            # MLA caches compressed latent representation: (d_c + d_h_R) per layer
            return self.layers * self.mla_latent_dim * bytes_per_elem
        # Standard MHA / GQA formula: 2 (Key + Value) * layers * kv_heads * head_dim * bytes
        return 2 * self.layers * self.kv_heads * self.head_dim * bytes_per_elem

    def weights_vram_gb(self, bytes_per_elem: int = 2) -> float:
        return (self.param_count_billions * 1e9 * bytes_per_elem) / (1024 ** 3)

    def kv_cache_vram_gb(self, seq_len: int, batch_size: int = 1, bytes_per_elem: int = 2) -> float:
        total_bytes = self.kv_bytes_per_token(bytes_per_elem) * seq_len * batch_size
        return total_bytes / (1024 ** 3)

# Define concrete enterprise model architectures
models = [
    ModelConfig(
        name="Llama-2-70B (MHA)",
        layers=80, query_heads=64, kv_heads=64, head_dim=128,
        param_count_billions=70.0, attention_type="MHA"
    ),
    ModelConfig(
        name="Llama-3.1-70B (GQA)",
        layers=80, query_heads=64, kv_heads=8, head_dim=128,
        param_count_billions=70.0, attention_type="GQA"
    ),
    ModelConfig(
        name="DeepSeek-V3-671B (MLA)",
        layers=61, query_heads=128, kv_heads=128, head_dim=128,
        param_count_billions=671.0, attention_type="MLA", mla_latent_dim=576
    ),
]

context_windows = [4_096, 32_768, 131_072]  # 4K, 32K, 128K
concurrency_levels = [1, 16, 64]

rows = []
for m in models:
    bytes_per_tok = m.kv_bytes_per_token(bytes_per_elem=2)
    for ctx in context_windows:
        for b in concurrency_levels:
            kv_gb = m.kv_cache_vram_gb(seq_len=ctx, batch_size=b, bytes_per_elem=2)
            rows.append({
                "Model": m.name,
                "Attention": m.attention_type,
                "Bytes/Token (FP16)": f"{bytes_per_tok:,} B",
                "Context Window": f"{ctx // 1024}K ({ctx:,})",
                "Batch Size": b,
                "KV Cache VRAM (GB)": round(kv_gb, 2),
            })

kv_df = pd.DataFrame(rows)
# Filter for key representative snapshot to display
snapshot_df = kv_df[(kv_df["Batch Size"] == 16) | ((kv_df["Context Window"] == "128K (131,072)") & (kv_df["Batch Size"] == 1))]
display(snapshot_df.reset_index(drop=True))

### Hardware Physics: Memory Bandwidth & Decode TPS Ceiling

Why is LLM generation memory-bandwidth bound? During autoregressive decoding, every single token generation step requires the GPU to:
1. Stream all model weights from HBM into on-chip SRAM to execute linear matrix projections.
2. Stream the accumulated KV cache for the active batch to compute attention.

```text
Memory Moved Per Step (Bytes) = Model_Weight_Bytes + KV_Cache_Bytes_for_Active_Batch
Theoretical Decode TPS Ceiling = Memory_Bandwidth_Bytes_sec ÷ Memory_Moved_Per_Step
```

On an **NVIDIA H100 SXM (80 GB HBM3, 3,350 GB/s bandwidth)**, we can compute the absolute ceiling of tokens per second across batch sizes and context lengths:

In [ ]:
h100_bandwidth_gbs = 3350.0  # 3.35 TB/s in decimal GB/s
h100_vram_gb = 80.0

tps_results = []
llama3 = models[1]  # Llama-3.1-70B
model_weight_gb = llama3.weights_vram_gb(bytes_per_elem=2)  # FP16 ~ 130.4 GB (requires 2x H100 or tensor parallel TP=2)
# Assuming TP=2 setup (2x H100 SXM, aggregated bandwidth = 6,700 GB/s, 160 GB VRAM)
tp2_bandwidth_gbs = h100_bandwidth_gbs * 2

for ctx in [2_048, 8_192, 32_768, 65_536]:
    for b in [1, 8, 32, 64]:
        kv_gb = llama3.kv_cache_vram_gb(seq_len=ctx, batch_size=b, bytes_per_elem=2)
        total_bytes_moved_gb = model_weight_gb + kv_gb
        max_system_tps = (tp2_bandwidth_gbs / total_bytes_moved_gb) * b
        tps_per_user = max_system_tps / b
        tps_results.append({
            "Context Length": f"{ctx // 1024}K",
            "Batch Size": b,
            "Weight VRAM (GB)": round(model_weight_gb, 1),
            "KV Cache VRAM (GB)": round(kv_gb, 2),
            "Total VRAM Moved/Step (GB)": round(total_bytes_moved_gb, 2),
            "Aggregate System TPS": round(max_system_tps, 1),
            "Per-User TPS": round(tps_per_user, 1)
        })

tps_df = pd.DataFrame(tps_results)
display(tps_df[tps_df["Batch Size"].isin([1, 32])].reset_index(drop=True))

In [ ]:
# Visualizing KV Cache Footprint at 128K context across attention mechanisms
fig, ax = plt.subplots(figsize=(10, 5))

batch_sizes = [1, 8, 32, 64]
x = np.arange(len(batch_sizes))
width = 0.25

colors = ["#dc2626", "#2563eb", "#16a34a"]
for idx, m in enumerate(models):
    vrams = [m.kv_cache_vram_gb(seq_len=131_072, batch_size=b, bytes_per_elem=2) for b in batch_sizes]
    offset = (idx - 1) * width
    bars = ax.bar(x + offset, vrams, width, label=m.name, color=colors[idx])
    for bar in bars:
        height = bar.get_height()
        if height > 0.1:
            ax.text(bar.get_x() + bar.get_width() / 2, height + 2,
                    f"{height:.0f}G", ha='center', va='bottom', fontsize=8)

ax.axhline(80, color='black', linestyle='--', linewidth=1.5, label='1x H100 80GB VRAM Limit')
ax.axhline(160, color='purple', linestyle=':', linewidth=1.5, label='2x H100 160GB VRAM Limit')
ax.set_xlabel('Concurrency Batch Size (Simultaneous Streams)')
ax.set_ylabel('KV-Cache VRAM Required (GB) at 128K Context')
ax.set_title('KV-Cache VRAM Demand at 128K Context: MHA vs GQA vs MLA', fontsize=12, pad=12)
ax.set_xticks(x)
ax.set_xticklabels([f'Batch {b}' for b in batch_sizes])
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Part 3: The Inference Roofline Model: Arithmetic Intensity vs Hardware Ceilings

The **Roofline Model** (Williams, Waterman & Patterson, 2008) provides a visual blueprint of system performance by plotting **Attainable FLOP/s** against **Arithmetic Intensity** (FLOPs per byte of memory transferred).

### Mathematical Formulation:
```text
Attainable FLOP/s = min( Peak FLOP/s , Memory_Bandwidth × Arithmetic_Intensity )
Ridge Point (Balance Point) = Peak FLOP/s ÷ Memory_Bandwidth
```

- **To the left of the Ridge Point (Slanted Ceiling)**: The workload is **memory-bandwidth bound**. Compute units are idle waiting for tensors to stream from HBM. Increasing compute capacity yields zero speedup.
- **To the right of the Ridge Point (Flat Ceiling)**: The workload is **compute-bound**. Memory channels are saturated with reuse. Peak Tensor Core throughput is the governing limit.

### LLM Serving Phases on the Roofline:
1. **Prefill Phase (Prompt Ingestion)**: Processes all prompt tokens in parallel. Weights are reused across all prompt tokens `T`. Arithmetic intensity scales roughly as `T` FLOPs/byte (e.g. 512 to 4,096 FLOPs/byte), placing prefill firmly in the **compute-bound** regime.
2. **Decode Phase (Token Generation, Batch=1)**: Reads the entire model weights tensor just to emit a single token. Arithmetic intensity is ~1.0 to 1.5 FLOPs/byte, trapping it deep in the **memory-bandwidth bound** regime.
3. **Batched Decode (Batch=64)**: Shares weight reads across 64 concurrently generated tokens, elevating arithmetic intensity to ~64 FLOPs/byte and climbing toward the ridge point.

In [ ]:
# NVIDIA H100 SXM5 Hardware Parameters (Dense FP16/BF16 Tensor Core)
peak_tflops = 1979.0         # 1,979 TFLOPS
bandwidth_tb_s = 3.35        # 3.35 TB/s = 3,350 GB/s
bandwidth_gb_s = bandwidth_tb_s * 1000.0
ridge_point = (peak_tflops * 1e12) / (bandwidth_gb_s * 1e9)  # FLOPs per byte

intensities = np.logspace(-0.5, 4.0, 500)  # From ~0.3 to 10,000 FLOPs/byte
attainable_tflops = np.minimum(peak_tflops, (bandwidth_gb_s * intensities) / 1e3)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(intensities, attainable_tflops, 'b-', linewidth=2.5, label='NVIDIA H100 SXM Roofline')

# Mark the Ridge Point
ax.plot(ridge_point, peak_tflops, 'ro', markersize=8, label=f'Ridge Point ({ridge_point:.1f} FLOPs/byte)')
ax.axvline(ridge_point, color='gray', linestyle=':', alpha=0.6)

# Operational Workload Points
workloads = [
    ("Decode (Batch 1)", 1.2, (bandwidth_gb_s * 1.2) / 1e3, "#dc2626"),
    ("Decode (Batch 16)", 18.0, (bandwidth_gb_s * 18.0) / 1e3, "#d97706"),
    ("Decode (Batch 64)", 72.0, (bandwidth_gb_s * 72.0) / 1e3, "#ca8a04"),
    ("Prefill (Seq 512)", 512.0, min(peak_tflops, (bandwidth_gb_s * 512.0) / 1e3), "#059669"),
    ("Prefill (Seq 4096)", 2048.0, peak_tflops, "#2563eb"),
]

for label, intensity, tflops, color in workloads:
    ax.plot(intensity, tflops, marker='s', markersize=9, color=color, label=f"{label} ({intensity:.1f} F/B)")
    ax.annotate(f"  {label}", (intensity, tflops), fontsize=9, fontweight='bold', color=color,
                verticalalignment='center')

# Shaded Architectural Regimes
ax.axvspan(0.3, ridge_point, color='red', alpha=0.06, label='Memory-Bandwidth Bound Zone')
ax.axvspan(ridge_point, 10000, color='green', alpha=0.06, label='Compute-Bound Zone')

ax.set_xlabel('Arithmetic Intensity (FLOPs / Byte transferred)', fontsize=11)
ax.set_ylabel('Attainable Throughput (TFLOPS)', fontsize=11)
ax.set_title('The Inference Roofline Model: Prefill vs Autoregressive Decode Regimes', fontsize=13, pad=12)
ax.set_xlim(0.3, 10000)
ax.set_ylim(1, 3000)
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

## Architectural Summary & Platform Engineering Rules

1. **Never Assume 1 Word = 1 Token**: Non-English languages and structured code incur a 2x–5x tokenization inflation penalty. Always audit token consumption using tokenizers matching your deployment target (`cl100k_base`, `o200k_base`, or Llama BPE).
2. **KV Cache is the Real Concurrency Barrier**: Model weights occupy fixed memory, but the KV cache scales linearly with `batch_size × sequence_length`. GQA (8x reduction) and MLA (93% reduction) are critical architectural innovations to support 128K context windows without memory exhaustion.
3. **Prefill vs Decode Decoupling**: Prefill is compute-bound, whereas single-token decode is strictly bound by memory bandwidth. Production serving engines leverage continuous chunked prefill and PagedAttention to maintain high GPU utilization across both regimes.